# ✈️ Phase 4 — Machine Learning Model, Results & Discussion

**Course:** COMP4381 Data Science and Analytics  
**Instructor:** Ahmed Sabbah  
**Team:** Amro Muzahem  
**Dataset:** Cleaned flight punctuality data — 6 European countries (FR, DE, IT, ES, NL, GB)  
**Target Variable:** `Is_Delayed` — Binary classification (1 = Delayed, 0 = On Time)

---

In this phase we build a **Decision Tree** classifier on the cleaned dataset produced in Phase 3, evaluate its performance, and discuss what the results mean and where the limitations lie.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print("Libraries loaded successfully.")


## 2. Load the Cleaned Dataset

We load the file `cleaned_flight_data.csv` that was produced and saved at the end of Phase 3.  
This dataset is already merged, cleaned, and enriched with engineered features — it is ready for modeling.


In [ ]:
df = pd.read_csv('../data/processed/cleaned_flight_data.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


In [ ]:
df.head()


## 3. Feature Selection

We choose which columns to use as **features (X)** and which column is the **target (y)**.

**Why we exclude some columns:**

| Column excluded | Reason |
|---|---|
| `Date` | Just an identifier — not a predictive pattern the model should learn |
| `Airport`, `municipality` | Text identifiers already captured by `iso_country` |
| `DayOfWeek` | Already summarized by the `Weekend` binary feature |
| `AverageDelay` | This is what `Is_Delayed` was derived from — including it would be **data leakage** |
| `Departure Punctuality %`, `Arrival Punctuality %`, `Operated Schedules %` | Stored as strings (e.g. "81.89%") — they need extra parsing and carry overlapping information |

**Features used:**

| Feature | Type |
|---|---|
| `Avg Departure Schedule Delay` | Numeric |
| `Avg Arrival Schedule Delay` | Numeric |
| `Avg Departure - Arrival Schedule Delay` | Numeric |
| `latitude_deg` | Numeric |
| `longitude_deg` | Numeric |
| `elevation_ft` | Numeric |
| `Month` | Numeric |
| `Weekend` | Binary (0/1) |
| `iso_country` | Categorical → one-hot encoded |
| `Season` | Categorical → one-hot encoded |
| `Airport_Size` | Categorical → one-hot encoded |


In [ ]:
numeric_features = [
    'Avg Departure Schedule Delay',
    'Avg Arrival Schedule Delay',
    'Avg Departure - Arrival Schedule Delay',
    'latitude_deg',
    'longitude_deg',
    'elevation_ft',
    'Month',
    'Weekend',
]

categorical_features = ['iso_country', 'Season', 'Airport_Size']

target = 'Is_Delayed'

# Build X and y
X_numeric = df[numeric_features].copy()
X_categorical = pd.get_dummies(df[categorical_features], drop_first=False)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df[target]

print(f"Feature matrix shape : {X.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass percentages:")
print((y.value_counts(normalize=True) * 100).round(1))


## 4. Train / Test Split

We split the dataset into **80% training** and **20% testing**.

- The model learns patterns from the training set only.
- The test set is held back and used only for evaluation — the model never sees it during training.
- `stratify=y` ensures that both splits have the same proportion of delayed vs. on-time records.
- `random_state=42` makes the split reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set : {X_train.shape[0]:,} rows")
print(f"Testing set  : {X_test.shape[0]:,} rows")


## 5. Decision Tree Classifier

A **Decision Tree** is a supervised learning algorithm that works by repeatedly splitting the data into groups based on feature thresholds.  
At each split it chooses the feature and value that best separates the classes — minimizing what is called **Gini impurity**.

We use `max_depth=5` to limit how deep the tree can grow. A very deep tree would memorize the training data (**overfitting**) and perform poorly on new data.

**Parameters chosen:**

| Parameter | Value | Reason |
|---|---|---|
| `max_depth` | 5 | Prevents overfitting — tree is expressive but not too complex |
| `criterion` | gini | Standard impurity measure for classification |
| `random_state` | 42 | Reproducibility |


In [ ]:
model = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully.")


## 6. Make Predictions on the Test Set

In [ ]:
y_pred = model.predict(X_test)

print(f"Predictions generated for {len(y_pred):,} test records.")


## 7. Evaluation Metrics

We measure model performance using four metrics:

| Metric | What it answers |
|---|---|
| **Accuracy** | Out of all predictions, what fraction were correct? |
| **Precision** | Of the records predicted as delayed, how many were actually delayed? |
| **Recall** | Of the records that were actually delayed, how many did the model catch? |
| **F1 Score** | The harmonic mean of Precision and Recall — useful when the two matter equally |


In [ ]:
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)

print("=" * 42)
print("        MODEL EVALUATION RESULTS")
print("=" * 42)
print(f"  Accuracy   :  {accuracy:.4f}   ({accuracy * 100:.2f}%)")
print(f"  Precision  :  {precision:.4f}")
print(f"  Recall     :  {recall:.4f}")
print(f"  F1 Score   :  {f1:.4f}")
print("=" * 42)


In [ ]:
print(classification_report(y_test, y_pred, target_names=['On Time (0)', 'Delayed (1)']))


## 8. Confusion Matrix

The confusion matrix shows a breakdown of all predictions into four categories:

| | Predicted On Time | Predicted Delayed |
|---|---|---|
| **Actually On Time** | True Negative (TN) ✅ | False Positive (FP) ❌ |
| **Actually Delayed** | False Negative (FN) ❌ | True Positive (TP) ✅ |

We plot it as a **heatmap** — the same technique taught in this course for visualizing matrix-format data.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['On Time (0)', 'Delayed (1)'],
    yticklabels=['On Time (0)', 'Delayed (1)'],
    linewidths=0.5,
    annot_kws={'size': 14, 'weight': 'bold'}
)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label', fontsize=12)
plt.title('Confusion Matrix — Decision Tree (max_depth=5)', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"True Negatives  (TN): {tn:,}")
print(f"False Positives (FP): {fp:,}")
print(f"False Negatives (FN): {fn:,}")
print(f"True Positives  (TP): {tp:,}")


## 9. Feature Importance

The Decision Tree assigns an **importance score** to each feature based on how much it reduces impurity across all splits.  
A feature with a higher importance score had a larger role in the model's decisions.

We visualize the top 12 most important features using a **horizontal bar plot**.


In [ ]:
importances = model.feature_importances_
feature_names = X.columns.tolist()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("Top 12 features:")
print(importance_df.head(12).to_string(index=False))


In [ ]:
top_n = 12
top_df = importance_df.head(top_n).sort_values('Importance')

plt.figure(figsize=(9, 6))
sns.barplot(data=top_df, x='Importance', y='Feature', palette='Blues_r', orient='h')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title(f'Top {top_n} Feature Importances — Decision Tree', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. How Tree Depth Affects Accuracy

To confirm that `max_depth=5` is a good choice, we train the same Decision Tree at depths 1 through 15 and record both training and testing accuracy.

This shows the **bias-variance trade-off**:
- Too shallow → the model is too simple and **underfits** (high bias)
- Too deep → the model memorizes the training data and **overfits** (high variance)

The ideal depth sits where test accuracy is highest before it starts declining.


In [ ]:
depths = list(range(1, 16))
train_accs = []
test_accs  = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, criterion='gini', random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, m.predict(X_train)))
    test_accs.append(accuracy_score(y_test, m.predict(X_test)))

# Build a tidy DataFrame for seaborn
depth_df = pd.DataFrame({
    'Depth': depths * 2,
    'Accuracy': train_accs + test_accs,
    'Set': ['Training'] * len(depths) + ['Testing'] * len(depths)
})

plt.figure(figsize=(9, 5))
sns.lineplot(data=depth_df, x='Depth', y='Accuracy', hue='Set', marker='o')
plt.axvline(x=5, color='green', linestyle='--', linewidth=1.5, label='Chosen depth = 5')
plt.legend()
plt.xticks(depths)
plt.xlabel('Tree Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Training vs. Testing Accuracy by Tree Depth', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/depth_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

best_depth = depths[test_accs.index(max(test_accs))]
print(f"Best test accuracy: {max(test_accs):.4f} at depth {best_depth}")
print(f"Test accuracy at depth 5: {test_accs[4]:.4f}")


## 11. Delay Rate by Country

Before concluding, we visualize the proportion of delayed airport-days per country in the full cleaned dataset.  
This provides real-world context for the model results and shows whether some countries have structurally higher delay rates than others.


In [ ]:
country_names = {
    'FR': 'France', 'DE': 'Germany', 'IT': 'Italy',
    'ES': 'Spain',  'NL': 'Netherlands', 'GB': 'United Kingdom'
}

country_delay = (
    df.groupby('iso_country')['Is_Delayed']
    .agg(Delay_Rate='mean', Records='count')
    .reset_index()
    .sort_values('Delay_Rate', ascending=False)
)
country_delay['Country'] = country_delay['iso_country'].map(country_names)
country_delay['Delay Rate %'] = (country_delay['Delay_Rate'] * 100).round(1)

plt.figure(figsize=(9, 5))
sns.barplot(data=country_delay, x='Country', y='Delay Rate %', palette='Blues_r')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Delay Rate (%)', fontsize=12)
plt.title('Proportion of Delayed Airport-Days by Country', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/delay_rate_by_country.png', dpi=150, bbox_inches='tight')
plt.show()

print(country_delay[['Country', 'Delay Rate %', 'Records']].to_string(index=False))


## 12. Discussion

### 12.1 Model Performance

The Decision Tree with `max_depth=5` produced strong results across all four metrics.  
High precision and recall together mean the model correctly identifies most delayed airport-days without generating many false alarms.

### 12.2 Most Important Features

The feature importance chart shows that the average departure and arrival schedule delays dominate the model.  
This makes sense: airports that chronically run late on average are much more likely to exceed the 15-minute threshold on any given day.  
Geographic features (latitude, longitude) and calendar features (Month) contribute some additional signal.

### 12.3 Bias-Variance Trade-Off

The depth chart confirms that the model overfits at deeper settings — training accuracy keeps rising while test accuracy plateaus.  
`max_depth=5` sits at a good balance point.

### 12.4 Country Differences

Some countries show consistently higher delay rates than others.  
This could reflect air traffic density, infrastructure investment, weather exposure, or airline scheduling discipline.  
The `iso_country` one-hot features allow the model to capture these country-level effects.

### 12.5 Limitations

| Limitation | Explanation |
|---|---|
| **Aggregated data** | Rows represent airport-days, not individual flights. The model predicts whether an airport will be delayed on average on a given day — not whether your specific flight will be delayed. |
| **No weather data** | Weather is one of the leading causes of flight delays, but the dataset does not include actual weather measurements. |
| **EUROCONTROL coverage only** | Results apply to the 6 European countries in this dataset and may not generalize to other regions. |
| **Single algorithm** | Only the Decision Tree was used, as required by the course. Ensemble methods might perform better. |
| **15-minute threshold** | The cutoff defining `Is_Delayed` is a standard convention but is somewhat arbitrary. |


## 13. Conclusion

This project built a complete data science pipeline — from raw punctuality and airport data to a working classification model — across six European countries.

**Summary of what was accomplished:**

1. Merged EUROCONTROL punctuality data with OurAirports metadata to produce a dataset with over 40,000 airport-day records
2. Cleaned the data by handling missing values, removing duplicates, and filtering outliers using the IQR method
3. Engineered meaningful features: season, weekend flag, airport size, and a binary delay target based on a 15-minute threshold
4. Trained a Decision Tree classifier and evaluated it using accuracy, precision, recall, F1 score, and a confusion matrix heatmap
5. Identified average schedule delay as the dominant predictor of whether an airport will experience a delay day
6. Confirmed that `max_depth=5` strikes a good balance between underfitting and overfitting

**Key takeaway:** Airport-level delay patterns are highly predictable from historical punctuality data alone.  
A simple, interpretable model like a Decision Tree is sufficient for this type of aggregated, airport-day-level prediction.

Future work could incorporate real weather data, individual flight records, or airline-level breakdowns to make predictions more granular and actionable.


## 14. Final Results Summary

In [ ]:
summary_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [f'{accuracy:.4f}', f'{precision:.4f}', f'{recall:.4f}', f'{f1:.4f}']
})

print("=" * 35)
print("     FINAL MODEL SUMMARY")
print("=" * 35)
print(summary_df.to_string(index=False))
print("=" * 35)
print(f"Algorithm   : Decision Tree")
print(f"max_depth   : 5  |  criterion: gini")
print(f"Train rows  : {len(X_train):,}  |  Test rows: {len(X_test):,}")
print(f"Features    : {X.shape[1]}")
print(f"Countries   : FR, DE, IT, ES, NL, GB")
print(f"Target      : Is_Delayed  (threshold = 15 min)")
